In [ ]:
import csv
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# url = "https://www.reddit.com/r/slavelabour/new/"
url = "https://www.reddit.com/r/forhire/new/"

# Specify the subreddit URL
subreddit_url = url

# Set up the Chrome WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(subreddit_url)

# Accept Reddit's cookies if prompted (optional)
try:
    cookies_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Accept All')]")
    cookies_button.click()
except:
    pass

# Scroll the page a few times to load more posts
scroll_pause_time = 2
scroll_count = 2

for _ in range(scroll_count):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause_time)

# After scrolling, get the page source for Beautiful Soup
page_source = driver.page_source

# Close the Selenium driver
driver.quit()

# Parse the page source with Beautiful Soup
soup = BeautifulSoup(page_source, "html.parser")

# List to store the scraped data
scraped_data = []

# Find all post containers
posts = soup.find_all(id=lambda x: x and x.startswith("post-title-t3_"))

# Loop through each post container and extract data
for post in posts:
    # Extract the unique post ID from the title's id attribute
    post_id = post.get("id").replace("post-title-", "")
    
    # Get the title
    title = post.get_text(strip=True)
    
    # Find the flair using the extracted post ID
    # flair = soup.find(attrs={"post-id": post_id})
    # flair = soup.find("div", class_="flair-content [&_.flair-image]:align-bottom max-w-full overflow-hidden whitespace-nowrap text-ellipsis")
    # flairs = driver.find_elements(By.CLASS_NAME, "flair-content")
    # flair_text = flairs[0].text if flairs else "N/A"  # Adjust to get specific flair as needed
    # flair_text = flair.get_text(strip=True) if flair else "N/A"
    
    # Find the description content
    description_div = soup.find(id=f"{post_id}-post-rtjson-content")
    if description_div:
        paragraphs = description_div.find_all("p")
        description = "\n".join([p.get_text(strip=True) for p in paragraphs])
    else:
        description = "N/A"
    
    # Store the data in a dictionary
    post_data = {
        "Post ID": post_id,
        "Title": title,
        "Flair": 'N/A Flair Text - Manually Added',
        "Description": description
    }
    
    scraped_data.append(post_data)


for i in scraped_data:
    # Get the title
    title = i['Title']
    
    # Use regex to identify and tag the flair
    if re.search(r'\[For Hire\]', title, re.IGNORECASE):
        i['Flair'] = "For Hire"
    elif re.search(r'\[Hiring\]', title, re.IGNORECASE):
        i['Flair'] = "Hiring"
    else:
        i['Flair'] = "Other"

In [10]:
# for i in scraped_data:
#     print(i['Flair'])
#     print(i['Title'])
#     print(i['Description'][:50])
#     print('\n')
#     print("="*40)

# print(len(scraped_data))
print(scraped_data[52])

{'Post ID': 't3_1gn9gff', 'Title': '[Offer] Social Media Marketing/SEO Digital Marketing Services/Consultancy for as low as $80 a week', 'Flair': 'Other', 'Description': "[Offer] Social Media Marketing/SEO Digital Marketing Services/Consultancy for as low as $80 a week\nHi everyone, I'm a recent Marketing graduate, a Freelance SEO Content Writer and a Social Media Manager. I'm looking for someone, an entrepreneur who wants to hire a Social Media Manager for their business.\nWhat I can offer:\n-Content Creation: I'll craft all sorts of visual and video content based on marketing strategies I believe best suited for your business. From stories, posts, reels.\n-Copy/Content Writing: I'll take care of all the marketing copy, captions, emails, and even articles if you want.\n-Community Management: I'll build meaningful connections with your community, I'll respond to every comments and queries, I'll engage with them creatively.\n-Strategic Planning: I'll use all my marketing knowledge to he